In [11]:
import ollama
import re

In [12]:
SYSTEM_INSTRUCTION = """
You are Meta-Expert, an extremely clever expert with the unique ability to collaborate with multiple experts (such as Expert Problem Solver, Expert Mathematician, Expert Essayist, etc.) to tackle any task and solve any complex problems. Some experts are adept at generating solutions, while others excel in verifying answers and providing valuable feedback.

As Meta-Expert, your role is to oversee the communication between the experts, effectively using their skills to answer a given question while applying your own critical thinking and verification abilities.

To communicate with a expert, type its name (e.g., "Expert Linguist" or "Expert Puzzle Solver"), followed by a colon ":", and then provide a detailed instruction enclosed within triple quotes. For example:

Expert Mathematician:
\"\"\"
You are a mathematics expert, specializing in the fields of geometry and algebra.
Compute the Euclidean distance between the points (-2, 5) and (3, 7).
\"\"\"

Ensure that your instructions are clear and unambiguous, and include all necessary information within the triple quotes. You can also assign personas to the experts (e.g., "You are a physicist specialized in...").

Interact with only one expert at a time, and break complex problems into smaller, solvable tasks if needed. Each interaction is treated as an isolated event, so include all relevant details in every call.

If you or an expert finds a mistake in another expert's solution, ask a new expert to review the details, compare both solutions, and give feedback. You can request an expert to redo their calculations or work, using input from other experts. Keep in mind that all experts, except yourself, have no memory! Therefore, always provide complete information in your instructions when contacting them. Since experts can sometimes make errors, seek multiple opinions or independently verify the solution if uncertain. Before providing a final answer, always consult an expert for confirmation. Ideally, obtain or verify the final solution with two independent experts. However, aim to present your final answer within 15 rounds or fewer.

Refrain from repeating the very same questions to experts. Examine their responses carefully and seek clarification if required, keeping in mind they don't recall past interactions.

Present the final answer as follows:
>> FINAL ANSWER:
\"\"\"
[final answer]
\"\"\"

For multiple-choice questions, select only one option. Each question has a unique answer, so analyze the provided information carefully to determine the most accurate and appropriate response. Please present only one solution if you come across multiple options.
"""

In [13]:
#FUNCTION TO STORE HISTORY FOR META MODEL

def initialize_meta_history(user_query):
    return [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user", "content": f"Question: {user_query}\n\nLet's first come up with a list of experts you may want to consult for this problem."}
    ]


# eventually will add what expert was called , what instructions were provied to it and what was the response of experts

In [14]:
#FUNCTION TO SEE IF THE META MODEL HAVE ARRIVED AT ANSWER OR IT IS REQUESTING SOME EXPERT TO CALL AND IF THERE IS ANY EXPERT WHAT ARE INSTRUCTIONS
#FOR THAT EXPERT.

def parse_meta_output(text):  #text will be meta model output.
    """
    Parses the Meta Model's output to determine its next action.
    Returns a tuple: (action_type, name, instruction.)
    """
    # 1. First, check if the model has reached a final conclusion
    final_pattern = r">>\s*FINAL\s*ANSWER:\s*\"\"\"(.*?)\"\"\""
    final_match = re.search(final_pattern, text, re.DOTALL)  # searching the pattern.
    
    if final_match:
        return "final", None, final_match.group(1).strip()
        
    # 2. If no final answer, check if it is trying to call an expert
    expert_pattern = r"([a-zA-Z0-9\s]+):\s*\"\"\"(.*?)\"\"\""   #captures key label pair :- [someLabel: """some content here"""]
    expert_match = re.search(expert_pattern, text, re.DOTALL)
    
    if expert_match and "FINAL ANSWER" not in expert_match.group(1).upper():
        expert_name = expert_match.group(1).strip()
        instructions = expert_match.group(2).strip()
        return "expert", expert_name, instructions
        
    # 3. If the model messed up the formatting entirely
    return None, None, None

In [15]:
# TESTING THE FUNCTION CREATED 
test_expert_string = """
This is a complex problem regarding model deployment. Let's ask a specialist.

Expert Machine Learning Engineer:
\"\"\"
You are an expert in deploying local LLMs. Please write a short curriculum outline covering Ollama CLI, the Python library, and REST API integration.
\"\"\"
"""

Output= parse_meta_output(test_expert_string)
print(Output)

('expert', 'Expert Machine Learning Engineer', 'You are an expert in deploying local LLMs. Please write a short curriculum outline covering Ollama CLI, the Python library, and REST API integration.')


In [16]:
test_final_string = """
After reviewing the outputs from the engineering expert, the solution is clear. Here is the final curriculum structure.

>> FINAL ANSWER:
\"\"\"
Module 1: Intro to Local AI
Module 2: RAG Workflows
Module 3: Multi-Model Orchestration
\"\"\"
"""

Output= parse_meta_output(test_final_string)
print(Output)

('final', None, 'Module 1: Intro to Local AI\nModule 2: RAG Workflows\nModule 3: Multi-Model Orchestration')


In [17]:
#FUNCTION TO CALL THE EXPERT THAT META MODEL DECIDED TO USE WITH THE INSTRUCTIONS.

def call_expert(expert_name, instructions, model="gpt-oss:120b-cloud"):
    """
    Calls the specialized expert using Ollama. 
    """
    print(f"\n⚙️ [Consulting {expert_name}...] \n")
    
    # We construct a completely fresh context for the expert.
    expert_messages = [
        {"role": "system", "content": f"You are {expert_name}. Follow the instructions provided carefully."},
        {"role": "user", "content": instructions}
    ]
    
    # Standard Ollama chat call
    response = ollama.chat(model=model, messages=expert_messages)
    expert_output = response['message']['content']
    
    return expert_output

In [18]:
# RUNNING EVERYTHING IN LOOP

def run_meta_prompting(user_query, max_iterations=15):
    """
    Executes the main meta-prompting loop.
    """
    print(f"Starting Meta-Prompting for query: '{user_query}'\n")
    
    # 1. Initialize the message history for the Meta Model
    meta_history = initialize_meta_history(user_query)
    
    for iteration in range(1, max_iterations + 1):
    
        # 2a. Prompt the Meta Model (The Conductor)
        print("🧠 [Meta Model is thinking...]")
        response = ollama.chat(model="gpt-oss:120b-cloud", messages=meta_history)
        meta_output = response['message']['content']
        
        # Add the Meta Model's output to the history so it remembers its own thoughts
        meta_history.append({"role": "assistant", "content": meta_output})
        
        # 2b & 2c. Parse the output in one go to check for final answer or expert call
        action, name, content = parse_meta_output(meta_output)
        
        if action == "final":
            print("\n✅ Meta Model has reached a final conclusion:\n")
            return content
            
        elif action == "expert":
            # Call the isolated expert (name = expert_name, content = instructions)
            expert_output = call_expert(name, content)
            print(f"💡 [{name} replied:]\n{expert_output}\n")
            
            # 2b (continued). Format the expert's output and append it to the Meta Model's history
            # The t_mid template from the paper:
            expert_feedback = f"Expert {name} provided the following output:\n\"\"\"\n{expert_output}\n\"\"\"\n\nBased on the information given, what are the most logical next steps or conclusions? Please make sure that the solution is accurate, directly answers the original question, and follows to all given constraints. Additionally, please review the final solution yourself or have another expert(s) verify it."
            
            meta_history.append({"role": "user", "content": expert_feedback})
            
        else:
            # 2d. Error Handling: The model didn't call an expert OR give a final answer.
            print("⚠️ [Formatting Error] Meta Model did not follow the required syntax.")
            error_msg = "Error: You must either call an expert using the 'Expert Name:\n\"\"\"\nInstructions\n\"\"\"' format, or provide the final answer using the '>> FINAL ANSWER:\n\"\"\"\n[final answer]\n\"\"\"' format. Please try again."
            meta_history.append({"role": "user", "content": error_msg})

    print("\n🛑 Reached maximum iterations without a final answer.")
    return None


In [19]:
query= "Use the numbers 3, 3, 8, and 8 exactly once to form a mathematical expression that evaluates to exactly 24. You can only  use addition, subtraction, multiplication, division, and parentheses."

In [20]:
final_result = run_meta_prompting(query)

Starting Meta-Prompting for query: 'Use the numbers 3, 3, 8, and 8 exactly once to form a mathematical expression that evaluates to exactly 24. You can only  use addition, subtraction, multiplication, division, and parentheses.'

🧠 [Meta Model is thinking...]
⚠️ [Formatting Error] Meta Model did not follow the required syntax.
🧠 [Meta Model is thinking...]

✅ Meta Model has reached a final conclusion:



In [21]:
print(final_result)

8 ÷ ( 3 − 8⁄3 ) = 24


In [22]:
query = """
A B2B SaaS platform is experiencing a sudden 15% month-over-month increase in churn among its mid-tier enterprise users. 
We track four key metrics: 'Login_Frequency', 'Support_Tickets_Opened', 'Feature_Adoption_Rate', and 'Subscription_Length'. 

Task 1: Identify the most appropriate statistical method to determine if 'Support_Tickets_Opened' has a statistically significant correlation with the 
recent churn spike, and explain how to interpret the p-value in this context.

Task 2: Assuming the statistical test proves a strong positive correlation, formulate a short, actionable business retention strategy to mitigate this
specific issue. 
"""

In [23]:
final_result = run_meta_prompting(query)

Starting Meta-Prompting for query: '
A B2B SaaS platform is experiencing a sudden 15% month-over-month increase in churn among its mid-tier enterprise users. 
We track four key metrics: 'Login_Frequency', 'Support_Tickets_Opened', 'Feature_Adoption_Rate', and 'Subscription_Length'. 

Task 1: Identify the most appropriate statistical method to determine if 'Support_Tickets_Opened' has a statistically significant correlation with the 
recent churn spike, and explain how to interpret the p-value in this context.

Task 2: Assuming the statistical test proves a strong positive correlation, formulate a short, actionable business retention strategy to mitigate this
specific issue. 
'

🧠 [Meta Model is thinking...]
⚠️ [Formatting Error] Meta Model did not follow the required syntax.
🧠 [Meta Model is thinking...]

⚙️ [Consulting Expert Statistician...] 

💡 [Expert Statistician replied:]
### 1. Which test / modelling approach is most appropriate?  

| Variable | Typical form in your data | What 

In [24]:
print(final_result)

**Task 1 – Appropriate statistical method & p‑value interpretation**

- **Method:** Use a **mixed‑effects logistic regression** (or logistic regression with clustered robust SEs) on individual‑level data (one row per user‑month).  
  - Model: `logit[P(churn=1)] = β0 + β1·log1p(Support_Tickets_Opened) + β2·Login_Frequency + β3·Feature_Adoption_Rate + β4·Subscription_Length + (1 | customer_id)`.  
  - This accounts for the binary churn outcome, treats ticket count as a (log‑transformed) numeric predictor, and controls for repeated observations of the same enterprise account.

- **Assumptions to verify:**  
  1. Independence of observations → use random intercept for each customer (or cluster‑robust SEs).  
  2. Linearity of the log‑odds with respect to the (possibly transformed) ticket count → check LOWESS plot; add splines or quadratic term if needed.  
  3. No perfect multicollinearity among covariates → compute VIFs (< 5).  
  4. Sufficient events per variable (≥ 10 churn events per p

In [49]:
query = """
A regional bank has recently deployed an XGBoost credit scoring model. Over the last quarter, the model has systematically rejected commercial loan 
applications from retail businesses in a specific emerging zip code. The model's key predictive features are 'Debt-to-Income_Ratio', 
'Local_Foot_Traffic_Index', 'Years_in_Operation', and 'Sector_Risk_Score'. 

Task 1: Determine the most appropriate Explainable AI (XAI) framework (e.g., SHAP, LIME, or Counterfactuals) to identify whether
the 'Local_Foot_Traffic_Index' feature is acting as a proxy for socioeconomic bias in these specific local rejections. Explain the 
technical justification for your choice.

Task 2: Based on the chosen XAI framework, outline a causal, stakeholder-centric communication strategy for the bank's Risk Management Board. 
The strategy must translate the technical XAI outputs into a managerial framework that justifies whether to override the algorithm or redesign the 
lending policy.
"""

In [50]:
final_result = run_meta_prompting(query)

Starting Meta-Prompting for query: '
A regional bank has recently deployed an XGBoost credit scoring model. Over the last quarter, the model has systematically rejected commercial loan 
applications from retail businesses in a specific emerging zip code. The model's key predictive features are 'Debt-to-Income_Ratio', 
'Local_Foot_Traffic_Index', 'Years_in_Operation', and 'Sector_Risk_Score'. 

Task 1: Determine the most appropriate Explainable AI (XAI) framework (e.g., SHAP, LIME, or Counterfactuals) to identify whether
the 'Local_Foot_Traffic_Index' feature is acting as a proxy for socioeconomic bias in these specific local rejections. Explain the 
technical justification for your choice.

Task 2: Based on the chosen XAI framework, outline a causal, stakeholder-centric communication strategy for the bank's Risk Management Board. 
The strategy must translate the technical XAI outputs into a managerial framework that justifies whether to override the algorithm or redesign the 
lending p

In [53]:
print(final_result)

**Task 1 – XAI framework selection**

✅ **Chosen framework:** **SHAP (Tree‑SHAP for XGBoost)**  

**Technical justification**  
- **Exact, model‑faithful attributions** for every tree split; guarantees that the contributions summed equal the model’s credit score.  
- **Global & local insight** – we can rank features across the whole portfolio and also drill‑down to individual loan decisions.  
- **Interaction detection** – `shap.interaction_values` quantifies pairwise effects (e.g., Local_Foot_Traffic_Index × ZIP‑code, Local_Foot_Traffic_Index × Median_Income).  
- **Proxy‑bias detection** – by aggregating SHAP values we can correlate the contribution of the foot‑traffic feature with external socioeconomic variables; a high correlation (>0.6) signals surrogate behavior.  
- **Stakeholder‑ready visualisations** – summary plots, force plots, dependence & interaction plots map directly to “points added/subtracted from the credit score,” which finance executives understand.  
- **Scalabili